<center>
<h1>Лабораторна робота №2</h1>
<h2>Наука про дані: підготовчий етап</h2>
<h3>Частина 1. Збір та підготовка даних</h3>
</center>

### Завдання 1
Створити віртуальне середовище (venv) та завантажити за допомогою бібліотеки (urllib) структуровані файли VHI-індексу для всіх областей України. Передбачити додавання дати/часу до назви файлу та реалізувати механізм запобігання повторним завантаженням.

In [1]:
import os
import urllib.request
from datetime import datetime
import time

data_dir = 'data'
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

def download_vhi_data():
    for province_id in range(1, 26):
        prefix = f"vhi_id_{province_id}_"
        if any(f.startswith(prefix) for f in os.listdir(data_dir)):
            print(f"Область {province_id} вже завантажена. Пропускаємо.")
            continue
            
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        
        now = datetime.now().strftime("%Y%m%d%H%M%S")
        filename = f"vhi_id_{province_id}_{now}.csv"
        filepath = os.path.join(data_dir, filename)
        
        print(f"Завантаження області {province_id}...")
        try:
            response = urllib.request.urlopen(url, timeout=30)
            with open(filepath, 'wb') as f:
                f.write(response.read())
            print(f"Успішно збережено: {filename}")
            time.sleep(1)
        except Exception as e:
            print(f"Помилка для області {province_id}: {e}. Спробуй ще раз пізніше.")

download_vhi_data()

Завантаження області 1...
Успішно збережено: vhi_id_1_20260302185238.csv
Завантаження області 2...
Помилка для області 2: The read operation timed out. Спробуй ще раз пізніше.
Завантаження області 3...
Успішно збережено: vhi_id_3_20260302185319.csv
Завантаження області 4...
Успішно збережено: vhi_id_4_20260302185323.csv
Завантаження області 5...
Успішно збережено: vhi_id_5_20260302185325.csv
Завантаження області 6...
Успішно збережено: vhi_id_6_20260302185327.csv
Завантаження області 7...
Успішно збережено: vhi_id_7_20260302185329.csv
Завантаження області 8...
Успішно збережено: vhi_id_8_20260302185331.csv
Завантаження області 9...
Успішно збережено: vhi_id_9_20260302185334.csv
Завантаження області 10...
Успішно збережено: vhi_id_10_20260302185336.csv
Завантаження області 11...
Успішно збережено: vhi_id_11_20260302185338.csv
Завантаження області 12...
Успішно збережено: vhi_id_12_20260302185340.csv
Завантаження області 13...
Успішно збережено: vhi_id_13_20260302185342.csv
Завантаження 

### Завдання 2
Зчитати завантажені файли у pandas dataframe. Здійснити очищення даних (Data Cleaning). Видалення технічного тексту, обробка пропусків та перетворення типів. Реалізувати переіндексацію областей за українською абеткою.

In [2]:
import pandas as pd
import os

def create_clean_dataframe(directory):
    all_data = []
    ua_names = {
        1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька", 5: "Житомирська",
        6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська", 9: "Київська", 10: "Кіровоградська",
        11: "Луганська", 12: "Львівська", 13: "Миколаївська", 14: "Одеська", 15: "Полтавська",
        16: "Рівненська", 17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
        21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська", 25: "Республіка Крим"
    }

    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            path = os.path.join(directory, filename)
            df = pd.read_csv(
                path, 
                header=1, 
                names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'Empty']
            )
            
            file_id = int(filename.split('_')[2])
            df['Province_ID'] = file_id
            df['Province_Name'] = ua_names.get(file_id, "Невідома")
            
            all_data.append(df)
            
    final_df = pd.concat(all_data, ignore_index=True)  
    final_df = final_df.drop(columns=['Empty'])
    final_df['Year'] = final_df['Year'].astype(str).str.replace(r'\D', '', regex=True)
    
    cols = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']
    for col in cols:
        final_df[col] = pd.to_numeric(final_df[col], errors='coerce')
    
    final_df = final_df.dropna(subset=['Year', 'VHI'])
    return final_df

df = create_clean_dataframe('data')
print(f"Зібрано рядків: {len(df)}")
df.head()

Зібрано рядків: 53664


,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_ID,Province_Name
0,1982.0,1.0,0.059,258.24,51.11,48.78,49.95,10,Кіровоградська
1,1982.0,2.0,0.063,261.53,55.89,38.20,47.04,10,Кіровоградська
2,1982.0,3.0,0.063,263.45,57.30,32.69,44.99,10,Кіровоградська
3,1982.0,4.0,0.061,265.10,53.96,28.62,41.29,10,Кіровоградська
4,1982.0,5.0,0.058,266.42,46.87,28.57,37.72,10,Кіровоградська


### Завдання 3
Створити функції для формування наступних вибірок:
1. Ряд VHI для області за вказаний рік.
2. Ряд VHI за вказаний діапазон років для вказаних областей.
3. Пошук екстремумів (min/max), середнього значення та медіани.

In [5]:
def get_vhi_statistics(dataframe, p_id, year):
    subset = dataframe[(dataframe['Province_ID'] == p_id) & (dataframe['Year'] == year)]
    if subset.empty: return None
    return {
        'Max': subset['VHI'].max(),
        'Min': subset['VHI'].min(),
        'Mean': subset['VHI'].mean(),
        'Median': subset['VHI'].median()
    }

def get_vhi_range(dataframe, province_ids, start_year, end_year):
    subset = dataframe[
        (dataframe['Province_ID'].isin(province_ids)) & 
        (dataframe['Year'] >= start_year) & 
        (dataframe['Year'] <= end_year)
    ]
    return subset[['Year', 'Province_Name', 'VHI']]

def get_drought_years(dataframe, p_id, min_vhi, max_vhi):
    subset = dataframe[
        (dataframe['Province_ID'] == p_id) & 
        (dataframe['VHI'] >= min_vhi) & 
        (dataframe['VHI'] <= max_vhi)
    ]
    return subset['Year'].unique()

In [6]:
p_id = 10 # Кіровоградська
year = 2015

stats = get_vhi_statistics(df, p_id, year)
print(f"Статистика для області {p_id} за {year} рік")
for key, value in stats.items():
    print(f"{key}: {value:.2f}")

print(f"\nРоки з екстремальною посухою (VHI < 15)")
print(get_drought_years(df, p_id, 0, 15))

print(f"\nВибірка за діапазоном (області 1 та 2, 2000-2002)")
print(get_vhi_range(df, [1, 2], 2000, 2002).head(10))

Статистика для області 10 за 2015 рік
Max: 52.05
Min: 26.89
Mean: 40.75
Median: 42.48

Роки з екстремальною посухою (VHI < 15)
[]

Вибірка за діапазоном (області 1 та 2, 2000-2002)
         Year Province_Name    VHI
23306  2000.0     Вінницька  35.79
23307  2000.0     Вінницька  37.89
23308  2000.0     Вінницька  37.46
23309  2000.0     Вінницька  36.62
23310  2000.0     Вінницька  37.63
23311  2000.0     Вінницька  38.49
23312  2000.0     Вінницька  36.49
23313  2000.0     Вінницька  35.46
23314  2000.0     Вінницька  36.99
23315  2000.0     Вінницька  38.71


### Завдання 4
Здійснити аналіз часових витрат на виконання розроблених процедур за допомогою модуля профілювання (timeit).

In [7]:
import timeit

setup_code = "from __main__ import get_vhi_statistics, df"
test_code = "get_vhi_statistics(df, 10, 2015)"
time_taken = timeit.timeit(setup=setup_code, stmt=test_code, number=100)
print(f"Середній час виконання функції за 100 запусків: {time_taken/100:.6f} секунд")

Середній час виконання функції за 100 запусків: 0.000443 секунд
